# Aula 01 — Introdução à Estatística com Python e dados do IBGE

**Curso:** Estatística Aplicada com Python usando dados públicos do IBGE/SIDRA
**Plataforma:** Google Colab

---

## Objetivos desta aula
1. Entender o que é Estatística e por que aprendê-la com dados reais.
2. Conhecer a API pública SIDRA do IBGE.
3. Fazer a primeira coleta de dados (IPCA mensal).
4. Inspecionar e preparar o `DataFrame` para análise.

---

## 1. O que é Estatística?

Estatística é a ciência de **coletar, organizar, analisar e interpretar dados** para
extrair conhecimento e apoiar decisões sob incerteza. Costuma-se dividi-la em:

| Ramo | O que faz | Exemplo |
|---|---|---|
| **Descritiva** | Resume e descreve dados observados. | Média do IPCA dos últimos 12 meses. |
| **Inferencial** | Generaliza de uma amostra para uma população. | Estimar o IPCA médio anual com IC 95%. |

Vamos usar dados **reais** do IBGE — nada de dados sintéticos. O Brasil oferece uma das
APIs estatísticas públicas mais completas do mundo: a **SIDRA**.

---

## 2. Por que IBGE/SIDRA?

- **Gratuita** e **sem chave de API**.
- Dados oficiais (IPCA, PIB, Censo, PNAD, etc.).
- Centenas de tabelas, milhares de variáveis.
- URL-driven: a consulta é montada na própria URL.

Padrão da URL:
```
https://apisidra.ibge.gov.br/values/t/{tabela}/n{nivel}/{cod}/v/{variavel}/p/{periodo}
```

In [ ]:
# === SETUP PARA GOOGLE COLAB ===
# Esta célula garante que o notebook funcione tanto em ambiente local
# quanto em quem clonar o repositório direto no Colab.

import sys, os, subprocess

# 1) Instala dependências (silencioso se já existirem)
pkgs = ["numpy", "pandas", "matplotlib", "seaborn", "scipy", "requests", "plotly", "statsmodels"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

# 2) Se estivermos no Colab e o repo ainda não foi clonado, clona.
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/220719/curso-estatistica-python.git"  # ajuste após criar o repo

if IN_COLAB and not os.path.exists("/content/curso-estatistica-python"):
    subprocess.run(["git", "clone", REPO_URL, "/content/curso-estatistica-python"], check=False)

# 3) Adiciona o diretório utils ao PYTHONPATH para importar o módulo SIDRA
BASE = "/content/curso-estatistica-python" if IN_COLAB else ".."
if BASE not in sys.path:
    sys.path.append(BASE)

# 4) Pasta de gráficos (cria se não existir)
GRAFICOS_DIR = os.path.join(BASE, "graficos")
os.makedirs(GRAFICOS_DIR, exist_ok=True)

print("✓ Ambiente pronto. Pasta de gráficos:", GRAFICOS_DIR)

## 3. Primeira coleta: IPCA mensal

O **IPCA** (Índice Nacional de Preços ao Consumidor Amplo) é o indicador oficial
de inflação do Brasil, calculado mensalmente pelo IBGE.

- Tabela SIDRA: `1737`
- Variável: `63` (variação mensal, em %)
- Nível: `n1` (Brasil)
- Período: últimos 60 meses

In [ ]:
# Importa a função utilitária do módulo do repositório
from utils.sidra_api import obter_ipca_mensal

# Coleta dos últimos 60 meses do IPCA (variação % mensal)
df_ipca = obter_ipca_mensal(n_ultimos=60)

# Visualiza as 5 primeiras linhas
df_ipca.head()

### 3.1 Inspecionando o DataFrame

Sempre que recebemos dados novos, devemos checar:
- Tamanho (`shape`)
- Tipos de cada coluna (`dtypes`)
- Valores ausentes (`isna().sum()`)
- Estatísticas rápidas (`describe()`)

In [ ]:
print("Formato:", df_ipca.shape)         # (linhas, colunas)
print("\nTipos:")
print(df_ipca.dtypes)
print("\nFaltantes:")
print(df_ipca.isna().sum())
print("\nResumo numérico:")
df_ipca.describe()

## 4. Primeira visualização

Antes de qualquer análise formal, **olhe os dados**. Um gráfico de linha temporal
revela tendências, sazonalidade e outliers.

In [ ]:
import matplotlib.pyplot as plt
import os

# Configuração estética padrão para todo o curso
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

fig, ax = plt.subplots()
ax.plot(df_ipca["data"], df_ipca["variacao_mensal"],
        marker="o", linewidth=1.5, color="#1f77b4")
ax.axhline(0, color="red", linestyle="--", linewidth=0.8, label="Inflação zero")
ax.set_title("IPCA — Variação mensal (%)", fontsize=14, fontweight="bold")
ax.set_xlabel("Data")
ax.set_ylabel("Variação (%)")
ax.legend()
plt.tight_layout()

# Salva na pasta de gráficos
plt.savefig(os.path.join(GRAFICOS_DIR, "aula01_ipca_serie.png"), dpi=150, bbox_inches="tight")
plt.show()

## 5. Exercícios

1. Modifique `obter_ipca_mensal` para trazer **120 meses** (10 anos) e refaça o gráfico.
2. Qual foi o **maior** valor mensal do IPCA no período? Em que mês ocorreu?
3. Quantos meses tiveram **deflação** (valor negativo)?

> 💡 Use `df_ipca["variacao_mensal"].idxmax()` para localizar o índice do máximo.

---

## Resumo da aula

- Estatística = ciência de extrair conhecimento de dados.
- A API SIDRA do IBGE é uma fonte rica, pública e sem chave.
- Sempre inspecione um DataFrame antes de analisar.
- Visualize antes de calcular: gráficos revelam o que números resumem.

**Próxima aula:** Estatística Descritiva — medidas de tendência central e dispersão.